# 1. 环境配置

## 1.1 python 环境准备

In [1]:
! pip install openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 arxiv==2.3.1

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/90/7f/340847023184305a6378d75ec71e1dd38a942dfe71b7c29314b8fbe26948/arxiv-2.3.1-py3-none-any.whl (11 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/4e/eb/c96d64137e29ae17d83ad2552470bafe3a7a915e85434d9942077d7fd011/feedparser-6.0.12-py3-none-any.whl (81 kB)
  Using cached sgmllib3k-1.0.0-py3-none-any.whl

   ---------------------------------------- 3/3 [arxiv]



## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [2]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 实践代码

为了能够顺利的演示内置中间件的使用详情，这里我们使用一段简单的智能体代码演示：

In [3]:
from langchain_community.chat_models import ChatTongyi
import os
llm = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-turbo")

from langchain_community.agent_toolkits.load_tools import load_tools
tools = load_tools(["arxiv"])

from langgraph.checkpoint.memory import InMemorySaver 
memory = InMemorySaver()

from langchain.agents import create_agent
agent = create_agent(model=llm, 
                     tools=tools, 
                     system_prompt="You are a helpful assistant", 
                     checkpointer=memory)

result1 = agent.invoke({"messages": [{"role": "user", "content": "请使用 arxiv 工具查询论文编号 1605.08386"}]}, config={"configurable": {"thread_id": "user_1"}})
print(result1["messages"][-1].content)

论文编号 1605.08386 的信息如下：

- **发表日期**: 2016-05-26
- **标题**: Heat-bath random walks with Markov bases
- **作者**: Caprice Stanley, Tobias Windisch
- **摘要**: 研究了由有限集的允许移动构成的格点图，这些移动可以是任意长度。我们证明了在固定整数矩阵的纤维上，这些图的直径可以被常数从上方界。然后研究了这些图上的热浴随机游走的混合行为。我们还给出了移动集的显式条件，使得热浴随机游走（Glauber动力学的一种推广）在固定维度下是一个扩展器。


In [ ]:
from typing import TypedDict

class Skill(TypedDict):
    """A skill that can be progressively disclosed to the agent."""
    name: str
    description: str
    content: str


SKILLS: list[Skill] = [
    {
        "name": "sales_analytics",
        "description": "Database schema and business logic for sales data analysis including customers, orders, and revenue.",
        "content": """# Sales Analytics Schema

## Tables

### customers
- customer_id (PRIMARY KEY)
- name
- email
- signup_date
- status (active/inactive)
- customer_tier (bronze/silver/gold/platinum)

### orders
- order_id (PRIMARY KEY)
- customer_id (FOREIGN KEY -> customers)
- order_date
- status (pending/completed/cancelled/refunded)
- total_amount
- sales_region (north/south/east/west)

### order_items
- item_id (PRIMARY KEY)
- order_id (FOREIGN KEY -> orders)
- product_id
- quantity
- unit_price
- discount_percent

## Business Logic

**Active customers**: status = 'active' AND signup_date <= CURRENT_DATE - INTERVAL '90 days'

**Revenue calculation**: Only count orders with status = 'completed'. Use total_amount from orders table, which already accounts for discounts.

**Customer lifetime value (CLV)**: Sum of all completed order amounts for a customer.

**High-value orders**: Orders with total_amount > 1000

## Example Query

-- Get top 10 customers by revenue in the last quarter
SELECT
    c.customer_id,
    c.name,
    c.customer_tier,
    SUM(o.total_amount) as total_revenue
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
  AND o.order_date >= CURRENT_DATE - INTERVAL '3 months'
GROUP BY c.customer_id, c.name, c.customer_tier
ORDER BY total_revenue DESC
LIMIT 10;
""",
    },
    {
        "name": "inventory_management",
        "description": "Database schema and business logic for inventory tracking including products, warehouses, and stock levels.",
        "content": """# Inventory Management Schema

## Tables

### products
- product_id (PRIMARY KEY)
- product_name
- sku
- category
- unit_cost
- reorder_point (minimum stock level before reordering)
- discontinued (boolean)

### warehouses
- warehouse_id (PRIMARY KEY)
- warehouse_name
- location
- capacity

### inventory
- inventory_id (PRIMARY KEY)
- product_id (FOREIGN KEY -> products)
- warehouse_id (FOREIGN KEY -> warehouses)
- quantity_on_hand
- last_updated

### stock_movements
- movement_id (PRIMARY KEY)
- product_id (FOREIGN KEY -> products)
- warehouse_id (FOREIGN KEY -> warehouses)
- movement_type (inbound/outbound/transfer/adjustment)
- quantity (positive for inbound, negative for outbound)
- movement_date
- reference_number

## Business Logic

**Available stock**: quantity_on_hand from inventory table where quantity_on_hand > 0

**Products needing reorder**: Products where total quantity_on_hand across all warehouses is less than or equal to the product's reorder_point

**Active products only**: Exclude products where discontinued = true unless specifically analyzing discontinued items

**Stock valuation**: quantity_on_hand * unit_cost for each product

## Example Query

-- Find products below reorder point across all warehouses
SELECT
    p.product_id,
    p.product_name,
    p.reorder_point,
    SUM(i.quantity_on_hand) as total_stock,
    p.unit_cost,
    (p.reorder_point - SUM(i.quantity_on_hand)) as units_to_reorder
FROM products p
JOIN inventory i ON p.product_id = i.product_id
WHERE p.discontinued = false
GROUP BY p.product_id, p.product_name, p.reorder_point, p.unit_cost
HAVING SUM(i.quantity_on_hand) <= p.reorder_point
ORDER BY units_to_reorder DESC;
""",
    },
]

from langchain.agents.middleware import AgentState
from typing import NotRequired

class CustomState(AgentState):
    skills_loaded: NotRequired[list[str]]

from langgraph.types import Command
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage

@tool
def load_skill(skill_name: str, runtime: ToolRuntime) -> Command:
    """
    将指定技能（skill）的完整内容加载到智能体的上下文中。

    当需要处理某一类特定请求、并且必须了解该业务领域的
    详细说明、业务规则或操作规范时，使用此工具。

    参数：
        skill_name：要加载的技能名称
    """
    # 查找并返回对应的技能
    for skill in SKILLS:
        if skill["name"] == skill_name:
            skill_content = f"已加载技能：{skill_name}\n\n{skill['content']}"

            # 更新系统状态，用于记录已加载的技能
            return Command(
                update={
                    "messages": [
                        ToolMessage(
                            content=skill_content,
                            tool_call_id=runtime.tool_call_id,
                        )
                    ],
                    # 在 agent 的 state 中记录已加载的技能名称
                    "skills_loaded": [skill_name],
                }
            )

    # 未找到对应技能
    available = ", ".join(s["name"] for s in SKILLS)
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=f"未找到技能“{skill_name}”。可用技能包括：{available}",
                    tool_call_id=runtime.tool_call_id,
                )
            ]
        }
    )



@tool
def write_sql_query(
    query: str,
    vertical: str,
    runtime: ToolRuntime,
) -> str:
    """
    为指定的业务领域编写并校验一条 SQL 查询。

    该工具用于对 SQL 查询进行格式化与校验。
    在使用该工具之前，必须先加载对应的 skill，
    以确保已经理解该业务领域的数据库结构（schema）。

    参数说明：
        query：要编写/校验的 SQL 查询语句
        vertical：业务领域名称（sales_analytics 或 inventory_management）
    """
    # 从运行时状态中读取已经加载过的技能列表
    skills_loaded = runtime.state.get("skills_loaded", [])

    # 如果所需的业务领域 skill 尚未加载，则返回错误提示
    if vertical not in skills_loaded:
        return (
            f"错误：在编写 SQL 查询之前，必须先加载 '{vertical}' 技能，"
            f"以便理解对应的数据库结构。"
            f"请先使用 load_skill('{vertical}') 来加载该 schema。"
        )

    # 校验并格式化 SQL 查询（此处为示例实现）
    return (
        f"{vertical} 业务领域的 SQL 查询如下：\n\n"
        f"```sql\n{query}\n```\n\n"
        f"✓ 查询已通过 {vertical} schema 校验\n"
        f"可以安全地在数据库中执行。"
    )

from langchain.agents.middleware import ModelRequest, ModelResponse, AgentMiddleware
from langchain.messages import SystemMessage
from typing import Callable, NotRequired

class SkillMiddleware(AgentMiddleware):
    """一个用于将技能（skill）描述注入到系统提示词（system prompt）中的中间件。"""

    state_schema = CustomState  
    tools = [load_skill, write_sql_query]  

    def __init__(self):
        """初始化中间件，并基于 SKILLS 生成技能说明提示内容。"""
        # 从 SKILLS 列表中构建技能说明文本
        skills_list = []
        for skill in SKILLS:
            skills_list.append(
                f"- **{skill['name']}**：{skill['description']}"
            )
        # 将所有技能说明拼接成一段文本
        self.skills_prompt = "\n".join(skills_list)

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """同步执行：在模型调用前，将技能描述注入到 system prompt 中。"""
        # 构建技能说明的补充内容
        skills_addendum = (
            f"\n\n## 可用技能（Available Skills）\n\n{self.skills_prompt}\n\n"
            "当你需要处理某一类具体请求的详细信息时，"
            "请使用 load_skill 工具来加载对应技能的完整内容。"
        )

        # 将技能说明追加到系统消息的内容块中
        new_content = list(request.system_message.content_blocks) + [
            {"type": "text", "text": skills_addendum}
        ]

        # 构造新的系统消息
        new_system_message = SystemMessage(content=new_content)

        # 用新的 system message 覆盖原有请求中的 system message
        modified_request = request.override(system_message=new_system_message)

        # 将修改后的请求继续交给下一个处理器（模型）执行
        return handler(modified_request)
    
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.chat_models import ChatTongyi
import os
model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-turbo")

agent = create_agent(
    model,
    system_prompt=(
        "You are a SQL query assistant that helps users "
        "write queries against business databases."
    ),
    middleware=[SkillMiddleware()],  
    checkpointer=InMemorySaver(),
)

import uuid

# 为当前这次对话生成一个唯一的线程 ID
thread_id = str(uuid.uuid4())

# 构造本次调用的配置参数
# configurable.thread_id 用于让 agent / checkpointer
# 识别并区分不同的对话线程
config = {"configurable": {"thread_id": thread_id}}

# 向智能体提出一个 SQL 查询请求
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "请编写一条 SQL 查询，用于找出在最近一个月内，"
                    "下过金额超过 1000 美元订单的所有客户"
                ),
            }
        ]
    },
    config
)

# 打印整个对话过程中的所有消息
for message in result["messages"]:
    if hasattr(message, 'pretty_print'):
        message.pretty_print()
    else:
        print(f"{message.type}: {message.content}")


================================ Human Message =================================

请编写一条 SQL 查询，用于找出在最近一个月内，下过金额超过 1000 美元订单的所有客户
================================== Ai Message ==================================
Tool Calls:
  load_skill (call_ff0c133db2994271a61b4d)
 Call ID: call_ff0c133db2994271a61b4d
  Args:
    skill_name: sales_analytics
================================= Tool Message =================================
Name: load_skill

已加载技能：sales_analytics

# Sales Analytics Schema

## Tables

### customers
- customer_id (PRIMARY KEY)
- name
- email
- signup_date
- status (active/inactive)
- customer_tier (bronze/silver/gold/platinum)

### orders
- order_id (PRIMARY KEY)
- customer_id (FOREIGN KEY -> customers)
- order_date
- status (pending/completed/cancelled/refunded)
- total_amount
- sales_region (north/south/east/west)

### order_items
- item_id (PRIMARY KEY)
- order_id (FOREIGN KEY -> orders)
- product_id
- quantity
- unit_price
- discount_percent

## Business Logic

**Active 

In [11]:
from langchain.agents.middleware import ModelRequest, ModelResponse, AgentMiddleware
from langchain.messages import SystemMessage
from typing import Callable

class SkillMiddleware(AgentMiddleware):
    """一个用于将技能（skill）描述注入到系统提示词（system prompt）中的中间件。"""
    tools = [load_skill] # 将 load_skill 工具注册为该中间件可用的工具

    def __init__(self):
        """初始化中间件，并基于 SKILLS 生成技能说明提示内容。"""
        skills_list = [] # 从 SKILLS 列表中构建技能说明文本
        for skill in SKILLS:
            skills_list.append(f"- **{skill['name']}**：{skill['description']}")  
        self.skills_prompt = "\n".join(skills_list) # 将所有技能说明拼接成一段文本

    def wrap_model_call(self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
        """同步执行：在模型调用前，将技能描述注入到 system prompt 中。"""
        skills_addendum = (f"\n\n## 可用技能（Available Skills）\n\n{self.skills_prompt}\n\n"
            "当你需要处理某一类具体请求的详细信息时，"
            "请使用 load_skill 工具来加载对应技能的完整内容。") # 构建技能说明的补充内容
        new_content = list(request.system_message.content_blocks) + [
            {"type": "text", "text": skills_addendum}] # 将技能说明追加到系统消息的内容块中
        new_system_message = SystemMessage(content=new_content) # 构造新的系统消息
        # 用新的 system message 覆盖原有请求中的 system message
        modified_request = request.override(system_message=new_system_message)
        # 将修改后的请求继续交给下一个处理器（模型）执行
        return handler(modified_request)

In [12]:
@tool
def load_skill(skill_name: str, runtime: ToolRuntime) -> Command:
    """...省略工具介绍信息..."""
    # 查找并返回对应的技能
    for skill in SKILLS:
        if skill["name"] == skill_name:
            skill_content = f"已加载技能：{skill_name}\n\n{skill['content']}"
            # 更新系统状态，用于记录已加载的技能
            return Command(update={
                    "messages": [ToolMessage(
                            content=skill_content,
                            tool_call_id=runtime.tool_call_id,)],
                    # 在 agent 的 state 中记录已加载的技能名称
                    "skills_loaded": [skill_name]})
    # 未找到对应技能
    available = ", ".join(s["name"] for s in SKILLS)
    return Command(update={"messages": [
                ToolMessage(
                    content=f"未找到技能“{skill_name}”。可用技能包括：{available}",
                    tool_call_id=runtime.tool_call_id,)]})

In [ ]:
==================================[1m Ai Message [0m==================================
Tool Calls:
  write_sql_query (call_440d901500244c668fe219)
 Call ID: call_440d901500244c668fe219
  Args:
    query: SELECT DISTINCT c.customer_id, c.name FROM customers c JOIN orders o ON c.customer_id = o.customer_id WHERE o.status = 'completed' AND o.order_date >= CURRENT_DATE - INTERVAL '1 month' AND o.total_amount > 1000;
    vertical: sales_analytics
=================================[1m Tool Message [0m=================================
Name: write_sql_query

sales_analytics 业务领域的 SQL 查询如下：

```sql
SELECT DISTINCT c.customer_id, c.name FROM customers c JOIN orders o ON c.customer_id = o.customer_id WHERE o.status = 'completed' AND o.order_date >= CURRENT_DATE - INTERVAL '1 month' AND o.total_amount > 1000;
```

✓ 查询已通过 sales_analytics schema 校验
可以安全地在数据库中执行。
==================================[1m Ai Message [0m==================================

以下是针对最近一个月内下过金额超过 1000 美元订单的所有客户的 SQL 查询：

```sql
SELECT DISTINCT c.customer_id, c.name 
FROM customers c 
JOIN orders o ON c.customer_id = o.customer_id 
WHERE o.status = 'completed' 
  AND o.order_date >= CURRENT_DATE - INTERVAL '1 month' 
  AND o.total_amount > 1000;
```

此查询将返回满足条件的客户 ID 和姓名。如果需要进一步分析这些客户，可以在此基础上扩展查询。